In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn pandas numpy torch protobuf sentencepiece

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from transformers.modeling_outputs import SequenceClassifierOutput
from dataclasses import dataclass
from typing import Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
ds = load_dataset("ailsntua/QEvasion")

train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_full_df['clarity_label']
)

In [ ]:
MODEL_NAME = "FacebookAI/roberta-large"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

In [ ]:
def tokenize_function_train(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    
    return tokenized


def tokenize_function_test(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [-100] * len(examples["clarity_label"])
    
    return tokenized


train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

train_tokenized = train_dataset.map(
    tokenize_function_train, 
    batched=True, 
    remove_columns=train_dataset.column_names
)
val_tokenized = val_dataset.map(
    tokenize_function_train, 
    batched=True, 
    remove_columns=val_dataset.column_names
)
test_tokenized = test_dataset.map(
    tokenize_function_test, 
    batched=True, 
    remove_columns=test_dataset.column_names
)

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadRobertaForClassification(RobertaPreTrainedModel):    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        
        self.roberta = RobertaModel(config, add_pooling_layer=True)
        
        classifier_dropout = (
            config.classifier_dropout 
            if config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        
        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)
        
        self.post_init()
    
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        clarity_labels: Optional[torch.LongTensor] = None,
        evasion_labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        
        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)
        
        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss_clarity = loss_fct(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
class MultiHeadDataCollator(DataCollatorWithPadding):    
    def __call__(self, features):
        clarity_labels = [f.pop('clarity_labels') for f in features] if 'clarity_labels' in features[0] else None
        evasion_labels = [f.pop('evasion_labels') for f in features] if 'evasion_labels' in features[0] else None
        
        batch = super().__call__(features)
        
        if clarity_labels is not None:
            batch['clarity_labels'] = torch.tensor(clarity_labels, dtype=torch.long)
        if evasion_labels is not None:
            batch['evasion_labels'] = torch.tensor(evasion_labels, dtype=torch.long)
        
        return batch

data_collator = MultiHeadDataCollator(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids
    
    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
    
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels
    
    valid_evasion_mask = labels_evasion != -100
    
    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')
    
    if valid_evasion_mask.sum() > 0:
        accuracy_evasion = accuracy_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask])
        f1_evasion = f1_score(labels_evasion[valid_evasion_mask], preds_evasion[valid_evasion_mask], average='macro')
    else:
        accuracy_evasion = 0.0
        f1_evasion = 0.0
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }

In [ ]:
class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())
        
        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)

In [ ]:
class EarlyStoppingCallback(TrainerCallback):
    
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
config = AutoConfig.from_pretrained(MODEL_NAME)

model = MultiHeadRobertaForClassification.from_pretrained(
    MODEL_NAME,
    config=config,
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_roberta_multihead",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_combined",
    greater_is_better=True,
    logging_steps=50,
    logging_strategy="steps",
    seed=SEED,
    report_to="none",
)

In [ ]:
trainer = MultiHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
)

In [ ]:
trainer.train()

In [ ]:
val_results = trainer.evaluate()
print("Validation Results")
print(f"\nClarity Head:")
print(f"  Accuracy: {val_results['eval_accuracy_clarity']:.4f}")
print(f"  Macro F1: {val_results['eval_f1_clarity']:.4f}")
print(f"\nEvasion Head:")
print(f"  Accuracy: {val_results['eval_accuracy_evasion']:.4f}")
print(f"  Macro F1: {val_results['eval_f1_evasion']:.4f}")
print(f"\nCombined F1: {val_results['eval_f1_combined']:.4f}")

In [ ]:
test_results = trainer.evaluate(test_tokenized)
print("Test Results")
print(f"\nClarity Head:")
print(f"  Accuracy: {test_results['eval_accuracy_clarity']:.4f}")
print(f"  Macro F1: {test_results['eval_f1_clarity']:.4f}")
print(f"\nEvasion Head:")
print(f"  Accuracy: {test_results['eval_accuracy_evasion']:.4f}")
print(f"  Macro F1: {test_results['eval_f1_evasion']:.4f}")
print(f"\nCombined F1: {test_results['eval_f1_combined']:.4f}")

In [ ]:
test_predictions = trainer.predict(test_tokenized)

logits_clarity, logits_evasion = test_predictions.predictions
preds_clarity = np.argmax(logits_clarity, axis=-1)
preds_evasion = np.argmax(logits_evasion, axis=-1)

pred_labels_clarity = [clarity_id2label[p] for p in preds_clarity]
pred_labels_evasion = [evasion_id2label[p] for p in preds_evasion]

y_true_clarity = test_df['clarity_label'].tolist()
y_true_evasion = test_df['evasion_label'].tolist()

print("Classification Report - Clarity Head")
print(classification_report(y_true_clarity, pred_labels_clarity, labels=clarity_labels, zero_division=0))

print("Classification Report - Evasion Head")
print(classification_report(y_true_evasion, pred_labels_evasion, labels=evasion_classes, zero_division=0))

In [ ]:
import os
import zipfile

os.makedirs("./predictions", exist_ok=True)


clarity_prediction_filename = "./predictions/prediction"  
with open(clarity_prediction_filename, 'w') as f:
    for pred in pred_labels_clarity:
        f.write(f"{pred}\n")

clarity_zip_filename = "../predictions/prediction_clarity_multihead.zip"
with zipfile.ZipFile(clarity_zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(clarity_prediction_filename, arcname="prediction")

evasion_prediction_filename = "./predictions/prediction"  # Reuse same filename
with open(evasion_prediction_filename, 'w') as f:
    for pred in pred_labels_evasion:
        f.write(f"{pred}\n")

evasion_zip_filename = "../predictions/prediction_evasion_multihead.zip"
with zipfile.ZipFile(evasion_zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(evasion_prediction_filename, arcname="prediction")
